
### 📝 TraceWhisperer 입문 튜토리얼: 특정 명령어의 실행 시점 찾기

**학습 목표:** 전력 파형(Power Trace)을 수집할 때, AES 암호화 알고리즘 내부의 특정 함수(예: `SubBytes`)가 정확히 어느 시점에 실행되는지 TraceWhisperer를 통해 찾아내고 시각화합니다.

#### 사전 하드웨어 준비 (매우 중요!)
이 코드를 실행하기 전에 타겟 보드(STM32F3)와 Husky 사이에 점퍼선 연결이 필요합니다.
* Target **TMS** 핀 ↔ Husky USERIO **D0** 핀
* Target **TCK** 핀 ↔ Husky USERIO **D1** 핀
* Target **TDO** 핀 ↔ Husky USERIO **D2** 핀

#### Step 1: 기본 설정 및 펌웨어 업로드
먼저 ChipWhisperer 장비를 연결하고, TraceWhisperer 기능을 활성화합니다.

In [ ]:
import chipwhisperer as cw
import time

# 1. 장비 및 타겟 설정
PLATFORM = 'CW308_STM32F3'
SCOPETYPE = 'OPENADC'

# 장비 연결 (Husky 기준)
scope = cw.scope()
target = cw.target(scope)

# 기본 클럭 및 통신 설정
scope.default_setup()
scope.adc.samples = 31000  # 파형을 충분히 길게 캡처하기 위해 샘플 수 증가
scope.gain.db = 12         # 전력 파형 증폭(Gain) 설정

# TraceWhisperer 기능 활성화
scope.trace.target = target
scope.trace.enabled = True

# 타겟 리셋 함수
def reset_target(scope):
    scope.io.nrst = 'low'
    time.sleep(0.05)
    scope.io.nrst = 'high'
    time.sleep(0.05)

reset_target(scope)
print("하드웨어 연결 완료!")

In [ ]:
%%bash -s "$PLATFORM"

cd simpleserial-trace/
make PLATFORM=$1 clean
make PLATFORM=$1

f=simpleserial-trace-CW308_STM32F3.elf

arm-none-eabi-objdump -D -x -s -S -l "$f" > "${f%.elf}.txt"

In [ ]:
# 2. 타겟 펌웨어 업로드
fw_path = 'simpleserial-trace/simpleserial-trace-CW308_STM32F3.hex'

prog = cw.programmers.STM32FProgrammer
cw.program_target(scope, prog, fw_path)
print("펌웨어 업로드 완료!")

#### Step 2: Trace 인터페이스 (SWO) 활성화
일반적으로 ARM 프로세서는 JTAG 모드로 부팅됩니다. Trace 데이터를 받기 위해서는 이를 SWD 모드로 전환하고 클럭 속도를 맞춰주어야 합니다.

In [ ]:
# 3. SWD 모드 전환 및 통신 속도 설정
scope.trace.clock.fe_clock_src = 'target_clock'  # 타겟 보드의 클럭을 기준 클럭으로 사용
scope.trace.trace_mode = 'SWO'                   # SWO 통신 모드 선택

# JTAG 모드에서 SWD 모드로 타겟 전환 (이 과정에서 앞서 연결한 점퍼선이 사용됩니다)
scope.trace.jtag_to_swd()

# SWO 통신 속도(Baud rate) 맞추기
acpr = 0  # 0이 가장 빠른 대역폭을 가집니다.
trigger_freq_mul = 8
scope.trace.clock.swo_clock_freq = scope.clock.clkgen_freq * trigger_freq_mul
scope.trace.target_registers.TPI_ACPR = acpr
scope.trace.swo_div = trigger_freq_mul * (acpr + 1)

# 상태 체크 (에러가 나면 하드웨어 연결을 다시 확인하세요)
assert scope.trace.clock.swo_clock_locked, "클럭이 동기화되지 않았습니다."
assert scope.userio.status & 0x4, "SWO 라인(D2)이 정상적으로 High 상태가 아닙니다." # <-- 이 부분이 수정되었습니다.
print("SWO 인터페이스 설정 완료!")

#### Step 3: 매칭 규칙 및 캡처 조건 설정
TraceWhisperer는 모든 동작을 기록할 수도 있지만(Raw mode), 데이터량이 너무 많아집니다. 초보자를 위해 우리가 "지정한 주소(PC 주소)"를 실행할 때만 기록하도록 설정합니다.

In [ ]:
# 주기적으로 보내는 Sync 프레임 비활성화 (노이즈 방지)
scope.trace.target_registers.DWT_CTRL = 0x40000021

# 펌웨어의 기본 트리거를 사용하도록 설정
scope.trace.capture.trigger_source = 'firmware trigger'

# Raw 모드 비활성화 (원하는 이벤트만 매칭하여 캡처)
scope.trace.capture.raw = False
scope.trace.set_pattern_match(0, [3, 8, 32])  # DWT_COMP 이벤트 패킷의 고정 헤더
scope.trace.capture.rules_enabled = [0]

# 트리거 라인이 High인 동안만 캡처 진행
scope.trace.capture.mode = 'while_trig'

# 매칭할 PC(Program Counter) 주소 설정
# 이 주소는 simpleserial-trace 펌웨어 기준입니다. (chipwhiserer 공식 깃허브의 hex 파일 기준)
# 0x080018c4: SubBytes() 함수의 시작 주소    -> 새 빌드 후 simpleserial-trace-CW308_STM32F3.txt로 확인한 주소  0x080017d8
# 0x0800188c: AddRoundKey() 함수의 시작 주소 -> 새 빌드 후 simpleserial-trace-CW308_STM32F3.txt로 확인한 주소  0x080017a4
scope.trace.set_isync_matches(addr0=0x080017d8, addr1=0x080017a4, match='both')

# 불필요한 주기적 PC 샘플링 끄기 (대역폭 확보를 위함)
scope.trace.set_periodic_pc_sampling(enable=0)
print("캡처 조건 및 타겟 주소 설정 완료!")

#### Step 4: 파형 수집 및 Trace 데이터 읽기
이제 실제로 암호화를 수행하면서 전력 파형과 Trace 데이터를 동시에 수집합니다.

In [ ]:
# Trace 스니퍼 무장 (캡처 준비)
scope.trace.arm_trace()

# 임의의 평문(Text)과 키(Key) 생성 후 1개의 파형 수집
ktp = cw.ktp.Basic()
key, text = ktp.next()
powertrace = cw.capture_trace(scope, target, text, key)

if powertrace is None:
    print("캡처 실패. 타겟 보드를 리셋하고 다시 시도해보세요.")
else:
    print("전력 파형 캡처 성공!")

# 장비의 버퍼에 쌓인 Trace 데이터 읽어오기
assert not scope.trace.fifo_empty(), "Trace 버퍼가 비어있습니다! (Trace 이벤트 발생 안함)"
raw_trace_data = scope.trace.read_capture_data()

# 매칭된 이벤트의 타임스탬프(시간 정보) 추출
times = scope.trace.get_rule_match_times(raw_trace_data, rawtimes=False, verbose=False)
print(f"총 {len(times)} 개의 매칭 이벤트를 찾았습니다!")

#### Step 5: 결과 시각화
수집한 전력 파형 위에 Trace 이벤트가 발생한 시점을 수직선으로 그려, 어느 부분에서 우리가 지정한 함수가 실행되었는지 눈으로 확인합니다. (Jupyter 환경에 친숙한 Bokeh 라이브러리 사용)

In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import Span

output_notebook()

# 그래프 생성
p = figure(width=1000, height=400, title="Power Trace & SubBytes/AddRoundKey Execution Points")

# 전력 파형 그리기 (빨간 선)
xrange = list(range(scope.adc.samples))
p.line(xrange, powertrace.wave, line_color="red")

# Trace 이벤트 시점 수직선으로 표시 (검은 선)
# 타겟 클럭과 ADC 클럭 사이의 비율(Multiplier)을 곱해주어 싱크를 맞춥니다.
multiplier = scope.clock.adc_mul
vlines = []
for t in times:
    vlines.append(Span(location=t[0] * multiplier, dimension='height', line_color='black', line_width=2))

p.renderers.extend(vlines)

show(p)

In [ ]:
scope.dis()
target.dis()

In [ ]:
isClean = True

if isClean:
    !cd simpleserial-trace && make PLATFORM=$PLATFORM clean && rm -f simpleserial-trace-CW308_STM32F3.txt

### 진행 팁 
1. **점퍼선 강조**: 가장 많이 실수하는 부분이 핀 연결입니다. 하드웨어 설정 단계에서 연결 상태를 꼭 재차 확인바랍니다.
2. **`fw_path` 수정**: 실습 환경에 따라 펌웨어 파일(`.hex`)의 위치가 다를 수 있습니다. 이 부분을 본인의 디렉토리 구조에 맞게 미리 세팅하시거나, 수정해주세요.
3. **주소값(`0x080018c4`)**: 해당 튜토리얼은 기본 제공되는 펌웨어 바이너리를 기준으로 합니다. 만약 직접 C 코드를 수정하고 새로 컴파일(make)한다면, 함수들의 메모리 시작 주소가 변경되므로 `objdump` 등을 통해 바뀐 주소를 찾아 입력해야 합니다.